<a href="https://colab.research.google.com/github/rijaldevanuj/atoms_GNN/blob/main/atoms_GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.7 MB/s  0:00:00


In [ ]:
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader

# 1. Download and load the QM9 dataset
dataset = QM9(root='./data/QM9')

# 2. Extract the very first molecule
data = dataset[0]
print(f"Number of atoms (nodes): {data.num_nodes}")
print(f"Number of bonds (edges): {data.num_edges}")
print(f"Node features shape: {data.x.shape}")

# 3. Create a DataLoader to handle batching
# We take a small subset (1,000 molecules) for faster testing
loader = DataLoader(dataset[:1000], batch_size=32, shuffle=True)

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Extracting data/QM9/raw/qm9_v3.zip
Processing...
Using a pre-processed version of the dataset. Please install 'rdkit' to alternatively process the raw data.
Done!


Number of atoms (nodes): 5
Number of bonds (edges): 8
Node features shape: torch.Size([5, 11])


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch.nn import Linear

class EquilibriumGNN(torch.nn.Module):
    def __init__(self, num_node_features):
        super(EquilibriumGNN, self).__init__()
        # Message Passing Layers
        self.conv1 = GCNConv(num_node_features, 64)
        self.conv2 = GCNConv(64, 64)

        # Final Readout Layer
        self.linear = Linear(64, 1)

    def forward(self, x, edge_index, batch):
        # 1. Message Passing & Activation
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)

        # 2. Pooling (Readout)
        x = global_mean_pool(x, batch)

        # 3. Final Output
        return self.linear(x)

# Instantiate the model using the 11 features from your previous output
model = EquilibriumGNN(num_node_features=11)

# Create a dummy batch vector [0, 0, 0, 0, 0] to tell the pooling layer
# that all 5 atoms belong to the same single graph
dummy_batch = torch.zeros(data.num_nodes, dtype=torch.int64)

# Perform a test forward pass
output = model(data.x, data.edge_index, dummy_batch)

print(f"Model output shape: {output.shape}")
print(f"Model output value: {output.item():.4f}")

Model output shape: torch.Size([1, 1])
Model output value: -0.0981


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
torch.save(model.state_dict(), '/content/drive/MyDrive/my_model.pth')


In [ ]:
from torch_geometric.loader import DataLoader
import torch.optim as optim
from torch.nn import L1Loss

# 1. Create a DataLoader (using a subset of 1000 molecules for speed)
train_loader = DataLoader(dataset[:1000], batch_size=32, shuffle=True)

# 2. Initialize optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = L1Loss() # Mean Absolute Error (MAE)

# Target property index for QM9 (Index 7 is Internal Energy)
target_idx = 7

# 3. Training Loop
model.train()
for epoch in range(5):
    total_loss = 0
    for batch in train_loader:
        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(batch.x, batch.edge_index, batch.batch)

        # Isolate the specific target property and match the tensor shape
        targets = batch.y[:, target_idx].view(-1, 1)

        # Compute loss
        loss = criterion(predictions, targets)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Accumulate total loss for the epoch
        total_loss += loss.item() * batch.num_graphs

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1} | Training MAE Loss: {avg_loss:.4f}")

Epoch 1 | Training MAE Loss: 7710.2294
Epoch 2 | Training MAE Loss: 7082.5902
Epoch 3 | Training MAE Loss: 3966.0483
Epoch 4 | Training MAE Loss: 937.6461
Epoch 5 | Training MAE Loss: 833.8785


In [ ]:
# 1. Create a test DataLoader with unseen molecules (graphs 1000 to 1100)
test_loader = DataLoader(dataset[1000:1100], batch_size=32, shuffle=False)

# 2. Set model to evaluation mode
model.eval()
test_loss = 0

# 3. Disable gradient calculation to save memory and compute
with torch.no_grad():
    for batch in test_loader:
        predictions = model(batch.x, batch.edge_index, batch.batch)
        targets = batch.y[:, target_idx].view(-1, 1)

        # Accumulate test loss
        loss = criterion(predictions, targets)
        test_loss += loss.item() * batch.num_graphs

avg_test_loss = test_loss / len(test_loader.dataset)
print(f"Test MAE Loss: {avg_test_loss:.4f}")

# 4. Perform a single prediction on a brand new molecule
sample = dataset[1000]
dummy_batch = torch.zeros(sample.num_nodes, dtype=torch.int64)
sample_pred = model(sample.x, sample.edge_index, dummy_batch)
sample_truth = sample.y[0, target_idx]

print(f"Sample Prediction: {sample_pred.item():.4f}")
print(f"Sample Ground Truth: {sample_truth.item():.4f}")

Test MAE Loss: 1469.3939
Sample Prediction: -8326.3564
Sample Ground Truth: -9237.8271


In [ ]:
import gradio as gr
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.nn import Linear

# 1. Define the architecture in the same cell
class ForcePredictorGNN(torch.nn.Module):
    def __init__(self, num_node_features):
        super(ForcePredictorGNN, self).__init__()
        self.conv1 = GCNConv(num_node_features, 64)
        self.conv2 = GCNConv(64, 64)
        self.linear = Linear(64, 3)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return self.linear(x)

# 2. Instantiate the model
model = ForcePredictorGNN(num_node_features=11)
model.eval()

# 3. Define the UI logic
def predict_forces_ui(molecule_index):
    sample = dataset[int(molecule_index)]
    with torch.no_grad():
        forces = model(sample.x, sample.edge_index)

    output_text = f"Analyzed Molecule {int(molecule_index)} ({sample.num_nodes} atoms).\n"
    output_text += f"Predicted 3D Forces (Fx, Fy, Fz):\n{forces.cpu().numpy()}"
    return output_text

# 4. Launch the interface
app = gr.Interface(
    fn=predict_forces_ui,
    inputs=gr.Number(label="Enter a Molecule Index (0 to 1000)", value=0),
    outputs=gr.Textbox(label="GNN Output"),
    title="Atomic Force Predictor",
)

app.launch(share=True, debug=True)

ModuleNotFoundError: No module named 'gradio'

In [ ]:
%%writefile app.py
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.nn import Linear
from torch_geometric.datasets import QM9
import gradio as gr

# 1. Load dataset
dataset = QM9(root='./data/QM9')

# 2. Define model architecture
class ForcePredictorGNN(torch.nn.Module):
    def __init__(self, num_node_features):
        super(ForcePredictorGNN, self).__init__()
        self.conv1 = GCNConv(num_node_features, 64)
        self.conv2 = GCNConv(64, 64)
        self.linear = Linear(64, 3)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return self.linear(x)

# 3. Instantiate model
model = ForcePredictorGNN(num_node_features=11)
model.eval()

# 4. Predict function
def predict_forces_ui(molecule_index):
    sample = dataset[int(molecule_index)]
    with torch.no_grad():
        forces = model(sample.x, sample.edge_index)

    output_text = f"Analyzed Molecule {int(molecule_index)} ({sample.num_nodes} atoms).\n"
    output_text += f"Predicted 3D Forces (Fx, Fy, Fz):\n{forces.cpu().numpy()}"
    return output_text

# 5. Interface definition
app = gr.Interface(
    fn=predict_forces_ui,
    inputs=gr.Number(label="Enter a Molecule Index (0 to 1000)", value=0),
    outputs=gr.Textbox(label="GNN Output"),
    title="Atomic Force Predictor",
)

if __name__ == "__main__":
    app.launch()

In [ ]:
# 1. Create requirements.txt
%%writefile requirements.txt
torch
torch_geometric
gradio

# 2. Create .gitignore to exclude dataset files and cache
%%writefile .gitignore
data/
*.zip
__pycache__/
.ipynb_checkpoints/
*.pt
*.pth

Writing requirements.txt
